In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import set_seed
from custom_heads_weighting import PositiveWeightedSetFitHead

from datasets import Dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from setfit import SetFitModel, Trainer, TrainingArguments 

from sklearn.metrics import(
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    multilabel_confusion_matrix
)

c:\Users\rebec\miniconda3\envs\setfit-thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#predetermined seed for later randomisation
SEED = 42 

#define text column
text_col = "Text"

#define label column
label_columns = [
    "Meaninglessness",
    "Loneliness",
    "Death Anxiety",
    "Death Acceptance",
    "Identity Confusion",
    "Freedom Responsibility",
    "Engagement",
    "Solitude"
]

In [ ]:
dev_df = pd.read_csv("Development_set.csv", encoding="utf-8-sig")

print("Development set shape:", dev_df.shape)
print("Development set columns:", dev_df.columns.tolist())


     Post_id                                               Text  \
0  POST_0046  Life is like a TV series that keeps getting re...   
1  POST_1526  The New Agora: Daily WWYD and light discussion...   
2  POST_0953  When exactly does free will happen? [SEP] To b...   
3  POST_0047  Archetypes (Jung, Hillman) vs existentialism a...   
4  POST_1357  How do you let go of someone you love? [SEP] I...   

   Death Anxiety  Death Acceptance  Loneliness  Solitude  Identity Confusion  \
0              0                 0           0         0                   0   
1              0                 0           0         0                   0   
2              0                 0           0         0                   0   
3              0                 0           0         0                   0   
4              0                 0           0         0                   0   

   Freedom Responsibility  Meaninglessness  Engagement  
0                       0                1           0  
1 

In [ ]:
# Rather code it so that it checks for whether all posts are either coded 0 or 1 and otherwise giving an error
def clean_label_value(value):

    if pd.isna(value):
        raise ValueError("Label value is NaN")
    
    if isinstance(value, str):
        value = value.strip()

    try:
        numeric_value = float(value)

    except Exception as err:
        raise ValueError(f"Unexpected label value: {value!r}") from err

        
    if numeric_value == 0:
        return 0
        
    elif numeric_value == 1:
        return 1
        
    else:
        raise ValueError (f"Label value must be 0 or 1, but found {value!r}")

In [ ]:
# Validate the label values and construct multilabel vectors
datasets = {
    "development": dev_df
}

for name, df in datasets.items():
    for col in label_columns:
        df[col] = df[col].apply(clean_label_value)

        invalid_mask = ~df[col].isin([0, 1])
        if invalid_mask.any():
            print(df.loc[invalid_mask, ["Post_id", col]])
            raise ValueError(f"Invalid label values found in column '{col}' of {name} set.")
        
    df["labels"] = df[label_columns].astype(int).values.tolist()

    print(f"{name.capitalize()} set label validation completed.")


development set:
                                                Text                    labels
0  Life is like a TV series that keeps getting re...  [1, 0, 0, 0, 0, 0, 0, 0]
1  The New Agora: Daily WWYD and light discussion...  [0, 0, 0, 0, 0, 0, 0, 0]
2  When exactly does free will happen? [SEP] To b...  [0, 0, 0, 0, 0, 0, 0, 0]
3  Archetypes (Jung, Hillman) vs existentialism a...  [0, 0, 0, 0, 0, 0, 0, 0]
4  How do you let go of someone you love? [SEP] I...  [0, 0, 0, 0, 0, 0, 0, 0]

test set:
                                                Text                    labels
0  I really hate when people invalidate a depress...  [0, 0, 0, 0, 0, 0, 0, 0]
1  A Stoic Parable to Memorize [SEP] I'm a dad wh...  [0, 0, 0, 0, 0, 1, 0, 0]
2  it will be no longer very soon [SEP] it, being...  [1, 0, 0, 0, 0, 0, 0, 0]
3  Does anyone else feel too weird for normal peo...  [0, 1, 0, 0, 0, 0, 0, 0]
4  Life has no meaning [SEP] There's no reason wh...  [1, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
# Check number of occurrences per label in the development set

print("\nDevelopment set:")

# Count appearances of each label
label_counts = dev_df[label_columns].sum()
print("Label counts: ", label_counts)

# Count posts with at least one label
labeled_posts = (dev_df[label_columns].sum(axis=1) > 0).sum()
print("Posts that contain at least one label: ", labeled_posts)

# Count posts with no label
posts_without_label = len(dev_df) - labeled_posts
print("Posts without any label: ", posts_without_label)

# Count all positive labels
labeled_total = dev_df[label_columns].sum().sum()
print("Total number of positive labels: ", labeled_total)


development set: 
Label counts:  Meaninglessness           34
Loneliness                23
Death Anxiety             22
Death Acceptance          11
Identity Confusion        12
Freedom Responsibility     7
Engagement                 7
Solitude                  12
dtype: int64
Posts that contain at least one label:  111
Posts without any label:  339
Total number of positive labels:  128

test set: 
Label counts:  Meaninglessness           4
Loneliness                3
Death Anxiety             2
Death Acceptance          1
Identity Confusion        1
Freedom Responsibility    1
Engagement                1
Solitude                  1
dtype: int64
Posts that contain at least one label:  14
Posts without any label:  36
Total number of positive labels:  14


In [39]:
def evaluate_setfit_model(model, dataset, dataset_name):
    texts = dataset[text_col]
    # Codes as provided by human coding
    y_true = np.array(dataset["labels"])

    # model's final binary decision per label
    y_pred = model.predict(texts)
  

    if torch.is_tensor(y_pred):
        y_pred = y_pred.detach().cpu().numpy()

    else:
        y_pred = np.asarray(y_pred)
        

    # Prediction scores per label
    y_proba = model.predict_proba(texts)

    if torch.is_tensor(y_proba):
        y_proba = y_proba.detach().cpu().numpy()

    else:
        y_proba = np.asarray(y_proba)

    results = {
        "dataset": dataset_name,
        "macro_f1": f1_score(y_true, y_pred, average = "macro", zero_division = 0),
        "micro_f1": f1_score(y_true, y_pred, average = "micro", zero_division = 0),
        "macro_precision": precision_score(y_true, y_pred, average = "macro", zero_division = 0),
        "micro_precision": precision_score(y_true, y_pred, average = "micro", zero_division = 0),
        "macro_recall": recall_score(y_true, y_pred, average = "macro", zero_division = 0),
        "micro_recall": recall_score(y_true, y_pred, average = "micro", zero_division = 0)
    }

    return results, y_true, y_pred, y_proba

In [40]:
def positive_weight_calculation(fold_train_df, label_columns):
# Calculating fold-specific positive class weights per label for BCEWithLogitsLoss
    pos_weights = []

    for l in label_columns:
        n_positive = fold_train_df[l].sum()
        if n_positive == 0:
            raise ValueError(f"No positive samples for label '{l}' in the training fold. A positive class weight cannot be computed.")
        
        else:
            n_negative = len(fold_train_df) - n_positive
            weight = n_negative / n_positive
        pos_weights.append(weight)
        print(f"Label: {l}, weight: {weight}")

    return pos_weights

In [ ]:
def square_root_weight_calculation(fold_train_df, label_columns):
# Calculating fold-specific square root positive class weights per label for BCEWithLogitsLoss
    pos_weights = []

    for l in label_columns:
        n_positive = fold_train_df[l].sum()
        if n_positive == 0:
            raise ValueError(f"No positive samples for label '{l}' in the training fold. A positive class weight cannot be computed.")
        
        else:
            n_negative = len(fold_train_df) - n_positive
            pos_weight = np.sqrt(n_negative / n_positive)
        pos_weights.append(pos_weight)
        print(f"Label: {l}, square root weight: {pos_weight}")

    return pos_weights

The following cells contain the different model-training configurations, which were evaluated during cross-validation. The evaluated backbone models were BAAI/bge-small-en-v1.5 (BGE) and Alibaba-NLP/gte-base-en-v1.5 (GTE).Furthermore, the configurations vary by backbone model (BAAI/bge-small-en-v1.5; Alibaba-NLP/gte-base-en-v1.5), sampling strategy (oversampling; undersampling) and weighting strategy (unweighted; positive weighted). The output files generated after each cross-validation run were manually renamed reflecting the corresponding backbone model name, sampling strategy and weighting strategy.

Comparing the sampling strategies undersampling vs oversampling with the model backbone BAAI/bge-small-en-v1.5

In [ ]:
# BAAI/bge-small-en-v1.5 undersampling unweighted 3-fold CV

# Creating the 3 folds for cross validation
X_dev = dev_df["Text"]

y_dev = dev_df[label_columns].to_numpy()

mskf = MultilabelStratifiedKFold(
    n_splits=3, 
    shuffle=True, 
    random_state=42
)

# Creating a list to collect the validation metrics
val_metrics = []

# Creating a list to collect the  metrics per label
label_metrics = []

# Creating a list to collect the prediction metrics
predictions = []

# For-loop for multilabel 3-fold cross-validation
for i, (train_index, validation_index) in enumerate(mskf.split(X_dev, y_dev)):
    fold_number = i + 1
    print("Fold ", fold_number)
    print("Training posts: ", len(train_index))
    print("Validation posts: ", len(validation_index))

    validation_labels = y_dev[validation_index]
    validation_counts = validation_labels.sum(axis=0)

    for j in range(len(label_columns)):
        print(label_columns[j], ": ", validation_counts[j])

    fold_train_df = dev_df.iloc[train_index].copy()
    fold_val_df = dev_df.iloc[validation_index].copy()

    fold_train_df.to_csv(f"Train_fold_{fold_number}.csv", index = False)
    fold_val_df.to_csv(f"Validation_fold_{fold_number}.csv", index = False)

    train_fold_dataset = Dataset.from_pandas(
    fold_train_df[[text_col, "labels"]].reset_index(drop = True)
    )

    validation_fold_dataset = Dataset.from_pandas(
    fold_val_df[[text_col, "labels"]].reset_index(drop = True)
    )

    # Define the settings for the model training
    args = TrainingArguments(
        output_dir = f"setfit_output/bge_undersampling_seed42_fold_{fold_number}",
        batch_size = (16, 2),
        num_epochs = (1, 16),
        sampling_strategy = "undersampling", 
        seed = SEED,
        show_progress_bar = True, 
    )

    num_classes = len(label_columns)

    # Reinitialise the random seed for reproducibility
    set_seed(SEED)
    
    # Initialise the SetFit Model and the selected Sentence Transformer model:"BAAI/bge-small-en-v1.5"
    model = SetFitModel.from_pretrained(
        "BAAI/bge-small-en-v1.5",
        multi_target_strategy="one-vs-rest", # multilabel differerntiable head with one binary output per label
        use_differentiable_head=True,
        head_params={"out_features": num_classes},
    )

    print("Model loaded. Train it before using for inference.")

    # Initialise the Trainer

    trainer = Trainer(
        model=model,
        args = args,
        train_dataset = train_fold_dataset,
        eval_dataset = validation_fold_dataset,
        column_mapping={
            "Text": "text",
            "labels": "label"
        }
    )

    # Starting the training of SetFit
    trainer.train()

    # Evaluation of the trained model - overall accuracy and overall f1-scores
    metrics = trainer.evaluate()
    print(metrics)

    # Initialising the evaluation of the validation dataset
    validation_results, y_true_val, y_pred_val, y_proba_val = evaluate_setfit_model(
        model,
        validation_fold_dataset,
        "validation"
    )

    print(validation_results)

    print("Actual positive label assignments:", y_true_val.sum())
    print("Predicted positive label assignments:", y_pred_val.sum())

    # Appending fold results to a list
    fold_results = validation_results.copy()
    fold_results["fold"] = fold_number
    fold_results["model"] = "BAAI/bge-small-en-v1.5"
    fold_results["sampling_strategy"] = "undersampling"
    fold_results["loss_function"] = "BCEWithLogitsLoss"
    fold_results["embedding_batch_size"] = 16
    fold_results["head_batch_size"] = 2
    fold_results["embedding_epochs"] = 1
    fold_results["head_epochs"] = 16    
    fold_results["seed"] = SEED


    val_metrics.append(fold_results)

    # Storing post-level validation predictions and probabilities in a list
    for idx in range(len(fold_val_df)):
        val_post = fold_val_df.iloc[idx]
        fold_post_metrics ={
            "model": "BAAI/bge-small-en-v1.5",
            "sampling_strategy": "undersampling",
            "loss_weighting": "unweighted",
            "fold": fold_number,
            "post_id": val_post["Post_id"]
        }

        for j, label in enumerate(label_columns):
            fold_post_metrics[f"true_{label}"] = y_true_val[idx][j]
            fold_post_metrics[f"pred_{label}"] = y_pred_val[idx][j]
            fold_post_metrics[f"prob_{label}"] = y_proba_val[idx][j] 

        predictions.append(fold_post_metrics)

    # Labelwise classificaion report
    label_report = classification_report(
        y_true_val,
        y_pred_val,
        target_names= label_columns,
        zero_division= 0,
        output_dict= True    
    )

    print(label_report)

    # Storing per-label metrics for the current fold in a list
    for l in label_columns:
        label_retrieval = label_report.get(l)
        fold_label_entry ={
            "model": "BAAI/bge-small-en-v1.5",
            "sampling_strategy": "undersampling",
            "loss_weighting": "unweighted",
            "fold": fold_number,
            "label": l,
            "precision": label_retrieval["precision"],#
            "recall": label_retrieval["recall"],
            "f1_score": label_retrieval["f1-score"],
            "support": label_retrieval["support"]
        }

        label_metrics.append(fold_label_entry)


# Convert lists into pd dataframes for csv file storage
val_metrics_df = pd.DataFrame(val_metrics)
label_metrics_df = pd.DataFrame(label_metrics)
predictions_df = pd.DataFrame(predictions)

# Store metrics dataframes as csv files 
val_metrics_df.to_csv("BGE_undersampling_unweighted_seed42_validation_fold_metrics.csv", index = False) 
label_metrics_df.to_csv("BGE_undersampling_unweighted_seed42_per_label_fold_metrics.csv", index = False)
predictions_df.to_csv("BGE_undersampling_unweighted_seed42_prediction_metrics.csv", index = False)   

In [ ]:
# BAAI/bge-small-en-v1.5 oversampling unweighted 3-fold CV

# Creating the 3 folds for cross validation
X_dev = dev_df["Text"]

y_dev = dev_df[label_columns].to_numpy()

mskf = MultilabelStratifiedKFold(
    n_splits=3, 
    shuffle=True, 
    random_state=42
)

# Creating a list to collect the validation metrics
val_metrics = []

# Creating a list to collect the  metrics per label
label_metrics = []

# Creating a list to collect the prediction metrics
predictions = []

# For-loop for multilabel 3-fold cross-validation
for i, (train_index, validation_index) in enumerate(mskf.split(X_dev, y_dev)):
    fold_number = i + 1
    print("Fold ", fold_number)
    print("Training posts: ", len(train_index))
    print("Validation posts: ", len(validation_index))

    validation_labels = y_dev[validation_index]
    validation_counts = validation_labels.sum(axis=0)

    for j in range(len(label_columns)):
        print(label_columns[j], ": ", validation_counts[j])

    fold_train_df = dev_df.iloc[train_index].copy()
    fold_val_df = dev_df.iloc[validation_index].copy()

    fold_train_df.to_csv(f"Train_fold_{fold_number}.csv", index = False)
    fold_val_df.to_csv(f"Validation_fold_{fold_number}.csv", index = False)

    train_fold_dataset = Dataset.from_pandas(
    fold_train_df[[text_col, "labels"]].reset_index(drop = True)
    )

    validation_fold_dataset = Dataset.from_pandas(
    fold_val_df[[text_col, "labels"]].reset_index(drop = True)
    )

    # Define the settings for the model training
    args = TrainingArguments(
        output_dir = f"setfit_output/bge_oversampling_seed42/fold_{fold_number}",
        batch_size = (16, 2),
        num_epochs = (1, 16),
        sampling_strategy = "oversampling", 
        seed = SEED,
        show_progress_bar = True, 
    )

    num_classes = len(label_columns)

    # Reinitialise the random seed for reproducibility
    set_seed(SEED)
    
    # Initialise the SetFit Model and the selected Sentence Transformer model:"BAAI/bge-small-en-v1.5"
    model = SetFitModel.from_pretrained(
        "BAAI/bge-small-en-v1.5",
        multi_target_strategy="one-vs-rest", # multilabel differerntiable head with one binary output per label
        use_differentiable_head=True,
        head_params={"out_features": num_classes},
    )

    print("Model loaded. Train it before using for inference.")

    # Initialise the Trainer

    trainer = Trainer(
        model=model,
        args = args,
        train_dataset = train_fold_dataset,
        eval_dataset = validation_fold_dataset,
        column_mapping={
            "Text": "text",
            "labels": "label"
        }
    )

    # Starting the training of SetFit
    trainer.train()

    # Evaluation of the trained model - overall accuracy and overall f1-scores
    metrics = trainer.evaluate()
    print(metrics)

    # Initialising the evaluation of the validation dataset
    validation_results, y_true_val, y_pred_val, y_proba_val = evaluate_setfit_model(
        model,
        validation_fold_dataset,
        "validation"
    )

    print(validation_results)

    print("Actual positive label assignments:", y_true_val.sum())
    print("Predicted positive label assignments:", y_pred_val.sum())

    # Appending fold results to a list
    fold_results = validation_results.copy()
    fold_results["fold"] = fold_number
    fold_results["model"] = "BAAI/bge-small-en-v1.5"
    fold_results["sampling_strategy"] = "oversampling"
    fold_results["loss_function"] = "BCEWithLogitsLoss"
    fold_results["embedding_batch_size"] = 16
    fold_results["head_batch_size"] = 2
    fold_results["embedding_epochs"] = 1
    fold_results["head_epochs"] = 16    
    fold_results["seed"] = SEED

    val_metrics.append(fold_results)

    # Storing post-level validation predictions and probabilities in a list
    for idx in range(len(fold_val_df)):
        val_post = fold_val_df.iloc[idx]
        fold_post_metrics ={
            "model": "BAAI/bge-small-en-v1.5",
            "sampling_strategy": "oversampling",
            "loss_weighting": "unweighted",
            "fold": fold_number,
            "post_id": val_post["Post_id"]
        }

        for j, label in enumerate(label_columns):
            fold_post_metrics[f"true_{label}"] = y_true_val[idx][j]
            fold_post_metrics[f"pred_{label}"] = y_pred_val[idx][j]
            fold_post_metrics[f"prob_{label}"] = y_proba_val[idx][j] 

        predictions.append(fold_post_metrics)

    # Labelwise classificaion report
    label_report = classification_report(
        y_true_val,
        y_pred_val,
        target_names= label_columns,
        zero_division= 0,
        output_dict= True    
    )

    print(label_report)

    # Storing per-label metrics for the current fold in a list
    for l in label_columns:
        label_retrieval = label_report.get(l)

        fold_label_entry ={
            "model": "BAAI/bge-small-en-v1.5",
            "sampling_strategy": "oversampling",
            "loss_weighting": "unweighted",
            "fold": fold_number,
            "label": l,
            "precision": label_retrieval["precision"],#
            "recall": label_retrieval["recall"],
            "f1_score": label_retrieval["f1-score"],
            "support": label_retrieval["support"]
        }

        label_metrics.append(fold_label_entry)


# Convert lists into pd dataframes for csv file storage
val_metrics_df = pd.DataFrame(val_metrics)
label_metrics_df = pd.DataFrame(label_metrics)
predictions_df = pd.DataFrame(predictions)

# Store metrics dataframes as csv files 
val_metrics_df.to_csv("BGE_oversampling_unweighted_seed42_validation_fold_metrics.csv", index = False) 
label_metrics_df.to_csv("BGE_oversampling_unweighted_seed42_per_label_fold_metrics.csv", index = False)
predictions_df.to_csv("BGE_oversampling_unweighted_seed42_prediction_metrics.csv", index = False)   

Comparing the backbones BAAI/bge-small-en-v1.5 vs Alibaba-NLP/gte-base-en-v1.5

In [ ]:
# Alibaba-NLP/gte-base-en-v1.5 oversampling unweighted 3-fold CV
# Creating the 3 folds for cross validation

X_dev = dev_df["Text"]

y_dev = dev_df[label_columns].to_numpy()

mskf = MultilabelStratifiedKFold(
    n_splits=3, 
    shuffle=True, 
    random_state=42
)

# Creating a list to collect the validation metrics
val_metrics = []

# Creating a list to collect the  metrics per label
label_metrics = []

# Creating a list to collect the prediction metrics
predictions = []

# For-loop for multilabel 3-fold cross-validation
for i, (train_index, validation_index) in enumerate(mskf.split(X_dev, y_dev)):
    fold_number = i + 1
    print("Fold ", fold_number)
    print("Training posts: ", len(train_index))
    print("Validation posts: ", len(validation_index))

    validation_labels = y_dev[validation_index]
    validation_counts = validation_labels.sum(axis=0)

    for j in range(len(label_columns)):
        print(label_columns[j], ": ", validation_counts[j])

    fold_train_df = dev_df.iloc[train_index].copy()
    fold_val_df = dev_df.iloc[validation_index].copy()
    fold_train_df.to_csv(f"Train_fold_{fold_number}.csv", index = False)
    fold_val_df.to_csv(f"Validation_fold_{fold_number}.csv", index = False)

    train_fold_dataset = Dataset.from_pandas(
    fold_train_df[[text_col, "labels"]].reset_index(drop = True)
    )

    validation_fold_dataset = Dataset.from_pandas(
    fold_val_df[[text_col, "labels"]].reset_index(drop = True)
    )

    # Define the settings for the model training
    args = TrainingArguments(
        output_dir = f"setfit_output/gte_unweighted_seed42/fold_{fold_number}",
        batch_size = (4, 2),
        num_epochs = (1, 16),
        sampling_strategy = "oversampling",
        seed = SEED,
        show_progress_bar = True, 
    )

    num_classes = len(label_columns)

    # Reinitialise the random seed for reproducibility
    set_seed(SEED)
    
    # Initialise the SetFit Model and the selected Sentence Transformer model: "Alibaba-NLP/gte-base-en-v1.5"
    model = SetFitModel.from_pretrained(
        "Alibaba-NLP/gte-base-en-v1.5",
        multi_target_strategy="one-vs-rest", # multilabel differerntiable head with one binary output per label
        use_differentiable_head=True,
        head_params={"out_features": num_classes},
        trust_remote_code=True,
    )

    # Initialise maximum sequence length for the model to 1536 tokens
    model.model_body.max_seq_length = 1536
    print(model.model_body.max_seq_length)

    print("Model loaded. Train it before using for inference.")

    # Initialise the Trainer
    trainer = Trainer(
        model=model,
        args = args,
        train_dataset = train_fold_dataset,
        eval_dataset = validation_fold_dataset,
        column_mapping={
            "Text": "text",
            "labels": "label"
        }
    )

    # Starting the training of SetFit
    trainer.train()

    # Evaluation of the trained model - overall accuracy and overall f1-scores
    metrics = trainer.evaluate()
    print(metrics)

    # Initialising the evaluation of the validation dataset
    validation_results, y_true_val, y_pred_val, y_proba_val = evaluate_setfit_model(
        model,
        validation_fold_dataset,
        "validation"
    )

    print(validation_results)
    print("Actual positive label assignments:", y_true_val.sum())
    print("Predicted positive label assignments:", y_pred_val.sum())

    # Appending fold results to a list
   
    fold_results = validation_results.copy()
    fold_results["fold"] = fold_number
    fold_results["model"] = "Alibaba-NLP/gte-base-en-v1.5"
    fold_results["sampling_strategy"] = "oversampling"
    fold_results["loss_function"] = "BCEWithLogitsLoss"
    fold_results["embedding_batch_size"] = 4
    fold_results["head_batch_size"] = 2
    fold_results["embedding_epochs"] = 1
    fold_results["head_epochs"] = 16   
    fold_results["seed"] = SEED

    val_metrics.append(fold_results)

    # Storing post-level validation predictions and probabilities in a list
    for idx in range(len(fold_val_df)):
        val_post = fold_val_df.iloc[idx]
        fold_post_metrics ={
            "model": "Alibaba-NLP/gte-base-en-v1.5",
            "sampling_strategy": "oversampling",
            "loss_weighting": "unweighted",
            "fold": fold_number,
            "post_id": val_post["Post_id"]
        }

        for j, label in enumerate(label_columns):
            fold_post_metrics[f"true_{label}"] = y_true_val[idx][j]
            fold_post_metrics[f"pred_{label}"] = y_pred_val[idx][j]
            fold_post_metrics[f"prob_{label}"] = y_proba_val[idx][j]

        predictions.append(fold_post_metrics)

    # Labelwise classificaion report
    label_report = classification_report(
        y_true_val,
        y_pred_val,
        target_names= label_columns,
        zero_division= 0,
        output_dict= True    
    )

    print(label_report)

    # Storing per-label metrics for the current fold in a list
    for l in label_columns:
        label_retrieval = label_report.get(l)

        fold_label_entry ={
            "model": "Alibaba-NLP/gte-base-en-v1.5",
            "sampling_strategy": "oversampling",
            "loss_weighting": "unweighted",
            "fold": fold_number,
            "label": l,
            "precision": label_retrieval["precision"],#
            "recall": label_retrieval["recall"],
            "f1_score": label_retrieval["f1-score"],
            "support": label_retrieval["support"]
        }

        label_metrics.append(fold_label_entry)


# Convert lists into pd dataframes for csv file storage
val_metrics_df = pd.DataFrame(val_metrics)
label_metrics_df = pd.DataFrame(label_metrics)
predictions_df = pd.DataFrame(predictions)

# Store metrics dataframes as csv files 
val_metrics_df.to_csv("Gte_oversampling_unweighted_seed42_validation_fold_metrics.csv", index = False) 
label_metrics_df.to_csv("Gte_oversampling_unweighted_seed42_per_label_fold_metrics.csv", index = False)
predictions_df.to_csv("Gte_oversampling_unweighted_seed42_prediction_metrics.csv", index = False)                   


Comparing unweighted vs positive weighted vs square root smoothing with the selected backbone Alibaba-NLP/gte-base-en-v1.5

In [ ]:
# Alibaba-NLP/gte-base-en-v1.5 oversampling positive weighted 3-fold CV

# Creating the 3 folds for cross validation
X_dev = dev_df["Text"]

y_dev = dev_df[label_columns].to_numpy()

mskf = MultilabelStratifiedKFold(
    n_splits=3, 
    shuffle=True, 
    random_state=42
)

# Creating a list to collect the validation metrics
val_metrics = []

# Creating a list to collect the  metrics per label
label_metrics = []

# Creating a list to collect the prediction metrics
predictions = []

# For-loop for multilabel 3-fold cross-validation
for i, (train_index, validation_index) in enumerate(mskf.split(X_dev, y_dev)):

    fold_number = i + 1

    print("Fold ", fold_number)
    print("Training posts: ", len(train_index))
    print("Validation posts: ", len(validation_index))

    validation_labels = y_dev[validation_index]
    validation_counts = validation_labels.sum(axis=0)


    for j in range(len(label_columns)):
        print(label_columns[j], ": ", validation_counts[j])

    fold_train_df = dev_df.iloc[train_index].copy()
    fold_val_df = dev_df.iloc[validation_index].copy()

    fold_train_df.to_csv(f"Train_fold_{fold_number}.csv", index = False)
    fold_val_df.to_csv(f"Validation_fold_{fold_number}.csv", index = False)

   
    train_fold_dataset = Dataset.from_pandas(
    fold_train_df[[text_col, "labels"]].reset_index(drop = True)
    )

    validation_fold_dataset = Dataset.from_pandas(
    fold_val_df[[text_col, "labels"]].reset_index(drop = True)
    )

    pos_weights = positive_weight_calculation(fold_train_df, label_columns)

    positive_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32)


    # Define the settings for the model training
    args = TrainingArguments(
        output_dir = f"setfit_output/gte_pos_weight_seed42/fold_{fold_number}",
        batch_size = (4, 2),
        num_epochs = (1, 16),
        sampling_strategy = "oversampling",
        seed = SEED,
        show_progress_bar = True, 
    )

    num_classes = len(label_columns)

    # Reinitialise the random seed for reproducibility
    set_seed(SEED)
    
    # Initialise the SetFit Model and the selected Sentence Transformer model: "Alibaba-NLP/gte-base-en-v1.5"
    model = SetFitModel.from_pretrained(
        "Alibaba-NLP/gte-base-en-v1.5",
        multi_target_strategy="one-vs-rest", # multilabel differentiable head with one binary output per label
        use_differentiable_head=True,
        head_params={"out_features": num_classes},
        trust_remote_code=True,
    )

    model.model_head = PositiveWeightedSetFitHead(
        in_features=model.model_head.in_features,
        out_features=model.model_head.out_features,
        multitarget=model.model_head.multitarget,
        positive_weights=positive_weights_tensor
    )

    model.model_head.to(model.device)
    # Initialise maximum sequence length for the model to 1536 tokens
    model.model_body.max_seq_length = 1536
    print(model.model_body.max_seq_length)

    print("Model loaded. Train it before using for inference.")

    # GPU / CPU device check
    print("Model device: ", model.device)
    print("Head weights device: ", model.model_head.positive_weights.device)
    print("Loss positive weights device: ", model.model_head.get_loss_fn().pos_weight.device)

    # Initialise the Trainer
    trainer = Trainer(
        model=model,
        args = args,
        train_dataset = train_fold_dataset,
        eval_dataset = validation_fold_dataset,
        column_mapping={
            "Text": "text",
            "labels": "label"
        }
    )

    # Starting the training of SetFit
    trainer.train()

    # Evaluation of the trained model - overall accuracy and overall f1-scores
    metrics = trainer.evaluate()
    print(metrics)

    # Initialising the evaluation of the validation dataset
    validation_results, y_true_val, y_pred_val, y_proba_val = evaluate_setfit_model(
        model,
        validation_fold_dataset,
        "validation"
    )

    print(validation_results)
    
    print("Actual positive label assignments:", y_true_val.sum())
    print("Predicted positive label assignments:", y_pred_val.sum())

    # Appending fold results to a list
    fold_results = validation_results.copy()
    fold_results["fold"] = fold_number
    fold_results["model"] = "Alibaba-NLP/gte-base-en-v1.5"
    fold_results["sampling_strategy"] = "oversampling"
    fold_results["loss_function"] = "pos_weighted_BCEWithLogitsLoss"
    fold_results["embedding_batch_size"] = 4
    fold_results["head_batch_size"] = 2
    fold_results["embedding_epochs"] = 1
    fold_results["head_epochs"] = 16   
    fold_results["seed"] = SEED

    val_metrics.append(fold_results)

    # Storing post-level validation predictions and probabilities in a list
    for idx in range(len(fold_val_df)):
        val_post = fold_val_df.iloc[idx]

        fold_post_metrics ={
            "model": "Alibaba-NLP/gte-base-en-v1.5",
            "sampling_strategy": "oversampling",
            "loss_weighting": "pos_weight",
            "fold": fold_number,
            "post_id": val_post["Post_id"]
        }

        for j, label in enumerate(label_columns):
            fold_post_metrics[f"true_{label}"] = y_true_val[idx][j]
            fold_post_metrics[f"pred_{label}"] = y_pred_val[idx][j]
            fold_post_metrics[f"prob_{label}"] = y_proba_val[idx][j] #CHECK if they are truly saved


        predictions.append(fold_post_metrics)

    # Labelwise classificaion report

    label_report = classification_report(
        y_true_val,
        y_pred_val,
        target_names= label_columns,
        zero_division= 0,
        output_dict= True,    
    )
    
    print(label_report)

    # Storing per-label metrics for the current fold in a list
    for l in label_columns:
        label_retrieval = label_report.get(l)

        fold_label_entry ={
            "model": "Alibaba-NLP/gte-base-en-v1.5",
            "sampling_strategy": "oversampling",
            "loss_weighting": "pos_weight",
            "fold": fold_number,
            "label": l,
            "precision": label_retrieval["precision"],
            "recall": label_retrieval["recall"],
            "f1_score": label_retrieval["f1-score"],
            "support": label_retrieval["support"]
        }

        label_metrics.append(fold_label_entry)



# Convert lists into pd dataframes for csv file storage
val_metrics_df = pd.DataFrame(val_metrics)
label_metrics_df = pd.DataFrame(label_metrics)
predictions_df = pd.DataFrame(predictions)

# Store metrics dataframes as csv files 
val_metrics_df.to_csv("Gte_oversampling_pos_weighted_seed42_validation_fold_metrics.csv", index = False) 
label_metrics_df.to_csv("Gte_oversampling_pos_weighted_seed42_per_label_fold_metrics.csv", index = False)
predictions_df.to_csv("Gte_oversampling_pos_weighted_seed42_prediction_metrics.csv", index = False)                   


Fold  1
Training posts:  300
Validation posts:  150
Meaninglessness :  11
Loneliness :  8
Death Anxiety :  7
Death Acceptance :  3
Identity Confusion :  4
Freedom Responsibility :  2
Engagement :  3
Solitude :  4
Label: Meaninglessness, weight: 12.043478260869565
Label: Loneliness, weight: 19.0
Label: Death Anxiety, weight: 19.0
Label: Death Acceptance, weight: 36.5
Label: Identity Confusion, weight: 36.5
Label: Freedom Responsibility, weight: 59.0
Label: Engagement, weight: 74.0
Label: Solitude, weight: 36.5


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
C:\Users\rebec\setfit\src\setfit\modeling.py:849: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "in_features": model_body.get_sentence_embedding_dimension(),


TypeError: PositiveWeightedSetFitHead.__init__() got an unexpected keyword argument 'pos_weights'

In [ ]:
# Alibaba-NLP/gte-base-en-v1.5 oversampling square root smoothing 3-fold CV

# Creating the 3 folds for cross validation
X_dev = dev_df["Text"]

y_dev = dev_df[label_columns].to_numpy()

mskf = MultilabelStratifiedKFold(
    n_splits=3, 
    shuffle=True, 
    random_state=42
)

# Creating a list to collect the validation metrics
val_metrics = []

# Creating a list to collect the  metrics per label
label_metrics = []

# Creating a list to collect the prediction metrics
predictions = []

# For-loop for multilabel 3-fold cross-validation
for i, (train_index, validation_index) in enumerate(mskf.split(X_dev, y_dev)):

    fold_number = i + 1

    print("Fold ", fold_number)
    print("Training posts: ", len(train_index))
    print("Validation posts: ", len(validation_index))

    validation_labels = y_dev[validation_index]
    validation_counts = validation_labels.sum(axis=0)


    for j in range(len(label_columns)):
        print(label_columns[j], ": ", validation_counts[j])

    fold_train_df = dev_df.iloc[train_index].copy()
    fold_val_df = dev_df.iloc[validation_index].copy()

    fold_train_df.to_csv(f"Train_fold_{fold_number}.csv", index = False)
    fold_val_df.to_csv(f"Validation_fold_{fold_number}.csv", index = False)

   
    train_fold_dataset = Dataset.from_pandas(
    fold_train_df[[text_col, "labels"]].reset_index(drop = True)
    )

    validation_fold_dataset = Dataset.from_pandas(
    fold_val_df[[text_col, "labels"]].reset_index(drop = True)
    )

    sq_root_weights = square_root_weight_calculation(
        fold_train_df, 
        label_columns
    )

    sq_root_weights_tensor = torch.tensor(
        sq_root_weights, 
        dtype=torch.float32
    )


    # Define the settings for the model training
    args = TrainingArguments(
        output_dir = f"setfit_output/gte_sq_root_weight_seed42/fold_{fold_number}",
        batch_size = (4, 2),
        num_epochs = (1, 16),
        sampling_strategy = "oversampling",
        seed = SEED,
        show_progress_bar = True, 
    )

    num_classes = len(label_columns)

    # Reinitialise the random seed for reproducibility
    set_seed(SEED)
    
    # Initialise the SetFit Model and the selected Sentence Transformer model: "Alibaba-NLP/gte-base-en-v1.5"
    model = SetFitModel.from_pretrained(
        "Alibaba-NLP/gte-base-en-v1.5",
        multi_target_strategy="one-vs-rest", # multilabel differentiable head with one binary output per label
        use_differentiable_head=True,
        head_params={"out_features": num_classes},
        trust_remote_code=True,
    )

    model.model_head = PositiveWeightedSetFitHead(
        in_features=model.model_head.in_features,
        out_features=model.model_head.out_features,
        multitarget=model.model_head.multitarget,
        positive_weights=sq_root_weights_tensor
    )

    model.model_head.to(model.device)
    # Initialise maximum sequence length for the model to 1536 tokens
    model.model_body.max_seq_length = 1536
    print(model.model_body.max_seq_length)

    print("Model loaded. Train it before using for inference.")

    # GPU / CPU device check
    print("Model device: ", model.device)
    print("Head weights device: ", model.model_head.positive_weights.device)
    print("Loss positive weights device: ", model.model_head.get_loss_fn().pos_weight.device)

    # Initialise the Trainer
    trainer = Trainer(
        model=model,
        args = args,
        train_dataset = train_fold_dataset,
        eval_dataset = validation_fold_dataset,
        column_mapping={
            "Text": "text",
            "labels": "label"
        }
    )

    # Starting the training of SetFit
    trainer.train()

    # Evaluation of the trained model - overall accuracy and overall f1-scores
    metrics = trainer.evaluate()
    print(metrics)

    # Initialising the evaluation of the validation dataset
    validation_results, y_true_val, y_pred_val, y_proba_val = evaluate_setfit_model(
        model,
        validation_fold_dataset,
        "validation"
    )

    print(validation_results)
    
    print("Actual positive label assignments:", y_true_val.sum())
    print("Predicted positive label assignments:", y_pred_val.sum())

    # Appending fold results to a list
    fold_results = validation_results.copy()
    fold_results["fold"] = fold_number
    fold_results["model"] = "Alibaba-NLP/gte-base-en-v1.5"
    fold_results["sampling_strategy"] = "oversampling"
    fold_results["loss_function"] = "square_root_weighted_BCEWithLogitsLoss"
    fold_results["embedding_batch_size"] = 4
    fold_results["head_batch_size"] = 2
    fold_results["embedding_epochs"] = 1
    fold_results["head_epochs"] = 16   
    fold_results["seed"] = SEED

    val_metrics.append(fold_results)

    # Storing post-level validation predictions and probabilities in a list
    for idx in range(len(fold_val_df)):
        val_post = fold_val_df.iloc[idx]

        fold_post_metrics ={
            "model": "Alibaba-NLP/gte-base-en-v1.5",
            "sampling_strategy": "oversampling",
            "loss_weighting": "square_root_smoothing",
            "fold": fold_number,
            "post_id": val_post["Post_id"]
        }

        for j, label in enumerate(label_columns):
            fold_post_metrics[f"true_{label}"] = y_true_val[idx][j]
            fold_post_metrics[f"pred_{label}"] = y_pred_val[idx][j]
            fold_post_metrics[f"prob_{label}"] = y_proba_val[idx][j] #CHECK if they are truly saved


        predictions.append(fold_post_metrics)

    # Labelwise classificaion report

    label_report = classification_report(
        y_true_val,
        y_pred_val,
        target_names= label_columns,
        zero_division= 0,
        output_dict= True,    
    )
    
    print(label_report)

    # Storing per-label metrics for the current fold in a list
    for l in label_columns:
        label_retrieval = label_report.get(l)

        fold_label_entry ={
            "model": "Alibaba-NLP/gte-base-en-v1.5",
            "sampling_strategy": "oversampling",
            "loss_weighting": "square_root_smoothing",
            "fold": fold_number,
            "label": l,
            "precision": label_retrieval["precision"],
            "recall": label_retrieval["recall"],
            "f1_score": label_retrieval["f1-score"],
            "support": label_retrieval["support"]
        }

        label_metrics.append(fold_label_entry)



# Convert lists into pd dataframes for csv file storage
val_metrics_df = pd.DataFrame(val_metrics)
label_metrics_df = pd.DataFrame(label_metrics)
predictions_df = pd.DataFrame(predictions)

# Store metrics dataframes as csv files 
val_metrics_df.to_csv("Gte_oversampling_square_root_weighted_seed42_validation_fold_metrics.csv", index = False) 
label_metrics_df.to_csv("Gte_oversampling_square_root_weighted_seed42_per_label_fold_metrics.csv", index = False)
predictions_df.to_csv("Gte_oversampling_square_root_weighted_seed42_prediction_metrics.csv", index = False)                   


Evaluating the models based on their mean macro-f1 and mean micro-f1

In [ ]:
# Evaluation of the BGE model based on mean macro-F1 (primary evaluation criterion) and mean micro-F1 (secondary evaluation criterion)
# Adapt parameters according to the model and settings being evaluated

cv_evaluation ={
    "model" : "BAAI/bge-small-en-v1.5",
    "sampling_strategy" : "undersampling", # adapt according to the selected settings
    "embedding_batch_size" : 16,
    "head_batch_size" : 2,
    "embedding_epochs" : 1,
    "head_epochs" : 16,
    "seed" : SEED,
    "mean_macro_f1" : val_metrics_df["macro_f1"].mean(),
    "sd_macro_f1" : val_metrics_df["macro_f1"].std(),
    "mean_micro_f1" : val_metrics_df["micro_f1"].mean(), 
    "sd_micro_f1" : val_metrics_df["micro_f1"].std()
}

cv_evaluation

cv_evaluation_df = pd.DataFrame([cv_evaluation])
cv_evaluation_df.to_csv("BGE_undersampling_seed42_cross_validation_evaluation.csv", index = False)

In [ ]:
# Evaluation of the GTE model based on mean macro-F1 (primary evaluation criterion) and mean micro-F1 (secondary evaluation criterion)
# Adapt parameters according to the model and settings being evaluated

cv_evaluation ={
    "model" : "Alibaba-NLP/gte-base-en-v1.5",
    "sampling_strategy" : "oversampling",
    "embedding_batch_size" : 4,
    "head_batch_size" : 2,
    "embedding_epochs" : 1,
    "head_epochs" : 16,
    "seed" : SEED,
    "mean_macro_f1" : val_metrics_df["macro_f1"].mean(),
    "sd_macro_f1" : val_metrics_df["macro_f1"].std(),
    "mean_micro_f1" : val_metrics_df["micro_f1"].mean(), 
    "sd_micro_f1" : val_metrics_df["micro_f1"].std()
}

cv_evaluation

cv_evaluation_df = pd.DataFrame([cv_evaluation])
cv_evaluation_df.to_csv("Gte_oversampling_seed42_cross_validation_evaluation.csv", index = False)

The following cells contain thresholding strategies: global threshold and micromacro.

In [13]:
def global_threshold(y_true, y_proba, candidate_threshold):
    """
    Function to calculate the global threshold for multilabel classification.
    
    Parameters:
    y_true (numpy.ndarray): True binary labels in binary indicator format.
    y_proba (numpy.ndarray): Predicted probabilities for each label.
    candidate_threshold (float): The threshold value to evaluate.
    
    Returns:
    float: The macro F1 score at the given threshold.
    """
    
    # Binarize predictions based on the candidate threshold
    y_pred = (y_proba >= candidate_threshold).astype(int)
    
    # Calculate the macro F1 score and micro F1 score
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    
    return macro_f1, micro_f1

In [14]:
# Global threshold optimization for the best macro-F1 score
prediction_df = pd.read_csv("GTE_oversampling_unweighted_prediction_metrics.csv")
print(prediction_df.columns)
prediction_df.head()

true_columns = [
    "true_Meaninglessness",
    "true_Loneliness",
    "true_Death Anxiety",
    "true_Death Acceptance",
    "true_Identity Confusion",
    "true_Freedom Responsibility",
    "true_Engagement",
    "true_Solitude"
]

prob_columns = [
    "prob_Meaninglessness",
    "prob_Loneliness",
    "prob_Death Anxiety",
    "prob_Death Acceptance",
    "prob_Identity Confusion",
    "prob_Freedom Responsibility",
    "prob_Engagement",
    "prob_Solitude"
]

y_true_val = prediction_df[true_columns].to_numpy()
y_proba_val = prediction_df[prob_columns].to_numpy()

print("True labels shape: ", y_true_val.shape)
print("Predicted probabilities shape: ", y_proba_val.shape)

print("Number of posts: ", prediction_df["post_id"].nunique())
print("Fold distribution:")
print(prediction_df["fold"].value_counts().sort_index())

print("Total number of rows: ", len(prediction_df))
print("Number of unique posts: ", prediction_df["post_id"].nunique())


threshold_interval = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
threshold_results = []

for t in threshold_interval:
    macro_f1, micro_f1 = global_threshold(y_true_val, y_proba_val, t)
    threshold_results.append((t, macro_f1, micro_f1))
    print(f"Threshold: {t}, Macro F1 Score: {macro_f1}, Micro F1 Score: {micro_f1}")

threshold_max = max(threshold_results, key=lambda x: x[1])
print(f"Optimal threshold: {threshold_max[0]}, Maximum Macro F1 Score: {threshold_max[1]}, Corresponding Micro F1 Score: {threshold_max[2]}")

threshold_results_df = pd.DataFrame(threshold_results, columns=["threshold", "macro_f1", "micro_f1"])
threshold_results_df.to_csv("Threshold_optimization_results.csv", index=False)

Index(['model', 'sampling_strategy', 'fold', 'post_id', 'true_Meaninglessness',
       'pred_Meaninglessness', 'prob_Meaninglessness', 'true_Loneliness',
       'pred_Loneliness', 'prob_Loneliness', 'true_Death Anxiety',
       'pred_Death Anxiety', 'prob_Death Anxiety', 'true_Death Acceptance',
       'pred_Death Acceptance', 'prob_Death Acceptance',
       'true_Identity Confusion', 'pred_Identity Confusion',
       'prob_Identity Confusion', 'true_Freedom Responsibility',
       'pred_Freedom Responsibility', 'prob_Freedom Responsibility',
       'true_Engagement', 'pred_Engagement', 'prob_Engagement',
       'true_Solitude', 'pred_Solitude', 'prob_Solitude'],
      dtype='str')
True labels shape:  (450, 8)
Predicted probabilities shape:  (450, 8)
Number of posts:  450
Fold distribution:
fold
1    150
2    150
3    150
Name: count, dtype: int64
Total number of rows:  450
Number of unique posts:  450
Threshold: 0.05, Macro F1 Score: 0.2877793949920522, Micro F1 Score: 0.3004291845493

In [15]:
# Fine search for optimal threshold
threshold_interval = [0.56, 0.57, 0.58, 0.59, 0.6, 0.61, 0.62, 0.63, 0.64]
threshold_results = []

for t in threshold_interval:
    macro_f1, micro_f1 = global_threshold(y_true_val, y_proba_val, t)
    threshold_results.append((t, macro_f1, micro_f1))
    print(f"Threshold: {t}, Macro F1 Score: {macro_f1}, Micro F1 Score: {micro_f1}")

threshold_max = max(threshold_results, key=lambda x: x[1])
print(f"Optimal threshold: {threshold_max[0]}, Maximum Macro F1 Score: {threshold_max[1]}, Corresponding Micro F1 Score: {threshold_max[2]}")

threshold_results_df = pd.DataFrame(threshold_results, columns=["threshold", "macro_f1", "micro_f1"])
threshold_results_df.to_csv("Threshold_optimization_fine_search.csv", index=False)

global_report = classification_report(y_true_val, (y_proba_val >= threshold_max[0]).astype(int), target_names=label_columns, zero_division=0, output_dict=True)
print(global_report)

Threshold: 0.56, Macro F1 Score: 0.3248313344926547, Micro F1 Score: 0.3782051282051282
Threshold: 0.57, Macro F1 Score: 0.32315859385712326, Micro F1 Score: 0.37540453074433655
Threshold: 0.58, Macro F1 Score: 0.327452545519496, Micro F1 Score: 0.3778501628664495
Threshold: 0.59, Macro F1 Score: 0.327452545519496, Micro F1 Score: 0.3778501628664495
Threshold: 0.6, Macro F1 Score: 0.329952545519496, Micro F1 Score: 0.3790849673202614
Threshold: 0.61, Macro F1 Score: 0.3309951267310851, Micro F1 Score: 0.380327868852459
Threshold: 0.62, Macro F1 Score: 0.3309951267310851, Micro F1 Score: 0.380327868852459
Threshold: 0.63, Macro F1 Score: 0.3105405812765396, Micro F1 Score: 0.37623762376237624
Threshold: 0.64, Macro F1 Score: 0.31212899636450603, Micro F1 Score: 0.3787375415282392
Optimal threshold: 0.61, Maximum Macro F1 Score: 0.3309951267310851, Corresponding Micro F1 Score: 0.380327868852459
{'Meaninglessness': {'precision': 0.2826086956521739, 'recall': 0.38235294117647056, 'f1-scor

In [8]:
epsilon = 1e-8

In [9]:
def micromacro(y_true_fold, y_proba_fold, j, cumulative_tp, cumulative_fp, cumulative_fn):

    # Calculate potential thresholds based on the label's probabilities in this fold
    probabilities = list(y_proba_fold)

    unique_probabilities = sorted(set(probabilities))

    threshold_candidates = []
    min_prob = min(unique_probabilities)
    min_threshold_candidate = min_prob - epsilon
    threshold_candidates.append(min_threshold_candidate)

    for i in range(len(unique_probabilities) - 1):
        left_neighbour = unique_probabilities[i]
        right_neighbour = unique_probabilities[i + 1]
        midpoint = (left_neighbour + right_neighbour) / 2

        threshold_candidates.append(midpoint)

    max_prob = max(unique_probabilities)
    max_threshold_candidate = max_prob + epsilon
    threshold_candidates.append(max_threshold_candidate)


    threshold_results = []

    for t in threshold_candidates:
        y_pred = (y_proba_fold >= t).astype(int)

        tp = 0
        fp = 0
        fn = 0

        # Calculate tp, fp, fn for the label in this fold with selected t
        for y_true, y_predicted in zip(y_true_fold, y_pred):

            if y_predicted == 1 and y_true == 1:
                tp += 1

            elif y_predicted == 1 and y_true == 0: 
                fp += 1

            elif y_predicted == 0 and y_true == 1:
                fn += 1 

        # Calculate F1 score
        denominator = 2 * tp + fp + fn
        
        if denominator == 0:
            f1 = 0
        
        else:
            f1 = (2 * tp) / denominator

         # Calculate micro-f1
        tp_total = cumulative_tp + tp
        fp_total = cumulative_fp + fp
        fn_total = cumulative_fn + fn
        
        micro_denominator = (2 * tp_total) + fp_total + fn_total
        if micro_denominator == 0:
            micro_f1 = 0
        
        else:
            micro_f1 = (2 * tp_total) / micro_denominator

        micromacro_objective = (1 / j) * f1 + micro_f1

        threshold_results.append((t, micromacro_objective, tp, fp, fn))

    best_micromacro = max(threshold_results, key=lambda x: x[1])
    best_threshold, best_objective, best_tp, best_fp, best_fn = best_micromacro

    return best_threshold, best_objective, best_tp, best_fp, best_fn

In [10]:
# Micromacro threshold strategy

prediction_df = pd.read_csv("GTE_oversampling_unweighted_prediction_metrics.csv")
print(prediction_df.columns)
prediction_df.head()

true_columns = [
    "true_Meaninglessness",
    "true_Loneliness",
    "true_Death Anxiety",
    "true_Death Acceptance",
    "true_Identity Confusion",
    "true_Freedom Responsibility",
    "true_Engagement",
    "true_Solitude"
]

prob_columns = [
    "prob_Meaninglessness",
    "prob_Loneliness",
    "prob_Death Anxiety",
    "prob_Death Acceptance",
    "prob_Identity Confusion",
    "prob_Freedom Responsibility",
    "prob_Engagement",
    "prob_Solitude"
]

y_true_val = prediction_df[true_columns].to_numpy()
y_proba_val = prediction_df[prob_columns].to_numpy()

# Restructure labels from highest to lowest prevalence of positive instances
# Check number of occurrences per label
restructured_label_order = []

label_position = 0

for l in true_columns:
 
    pos_label_count = prediction_df[l].sum()
    print("Label counts: ", pos_label_count)

    label_instance = (l, pos_label_count)

    restructured_label_order.append(label_instance)

restructured_label_order = sorted(restructured_label_order, key=lambda x: x[1], reverse=True)

cumulative_tp = 0
cumulative_fp = 0 
cumulative_fn = 0

optimal_label_thresholds = []

for label_info in restructured_label_order:
    label_position += 1   
    label_name = label_info[0]

    print("\nLabel:", label_name, "position:", label_position)

    fold_count_tp = []
    fold_count_fp = []
    fold_count_fn = []
    fold_thresholds = []

    for fold in sorted(prediction_df["fold"].unique()):
        fold_rows = prediction_df[prediction_df["fold"] == fold]
        y_true_fold = fold_rows[label_name]
        proba_name = label_name.replace("true_", "prob_")
        y_proba_fold = fold_rows[proba_name]


        best_threshold, best_objective, best_tp, best_fp, best_fn = micromacro(
            y_true_fold, 
            y_proba_fold, 
            label_position, 
            cumulative_tp, 
            cumulative_fp, 
            cumulative_fn
        )

        print(
            "fold:", fold,
            "threshold:", best_threshold,
            "micromacro objective:", best_objective,
            "TP:", best_tp,
            "FP:", best_fp,
            "FN:", best_fn
        )

        fold_thresholds.append(best_threshold)
        fold_count_tp.append(best_tp)
        fold_count_fp.append(best_fp)
        fold_count_fn.append(best_fn)

    best_mean_label_threshold = np.mean(fold_thresholds)

    print(
        label_name, 
        "fold thresholds:", fold_thresholds, 
        "mean threshold:", best_mean_label_threshold
    )

    optimal_label_threshold = (label_name, best_mean_label_threshold)
    optimal_label_thresholds.append(optimal_label_threshold)


    cumulative_tp += sum(fold_count_tp)
    cumulative_fp += sum(fold_count_fp)
    cumulative_fn += sum(fold_count_fn)    

    print(
        "cumulative TP:", cumulative_tp,
        "cumulative FP:", cumulative_fp,
        "cumulative FN:", cumulative_fn
    )

print(optimal_label_thresholds)

Index(['model', 'sampling_strategy', 'fold', 'post_id', 'true_Meaninglessness',
       'pred_Meaninglessness', 'prob_Meaninglessness', 'true_Loneliness',
       'pred_Loneliness', 'prob_Loneliness', 'true_Death Anxiety',
       'pred_Death Anxiety', 'prob_Death Anxiety', 'true_Death Acceptance',
       'pred_Death Acceptance', 'prob_Death Acceptance',
       'true_Identity Confusion', 'pred_Identity Confusion',
       'prob_Identity Confusion', 'true_Freedom Responsibility',
       'pred_Freedom Responsibility', 'prob_Freedom Responsibility',
       'true_Engagement', 'pred_Engagement', 'prob_Engagement',
       'true_Solitude', 'pred_Solitude', 'prob_Solitude'],
      dtype='str')
Label counts:  34
Label counts:  23
Label counts:  22
Label counts:  11
Label counts:  12
Label counts:  7
Label counts:  7
Label counts:  12

Label: true_Meaninglessness position: 1
fold: 1 threshold: 0.80775055 micromacro objective: 0.8333333333333334 TP: 5 FP: 8 FN: 6
fold: 2 threshold: 0.00019628148 micr

In [ ]:
# Evaluation of the mean micromacro thresholds

# Reorder the labels to fit the df ordering
threshold_dict = dict(optimal_label_thresholds)
threshold_vector = [threshold_dict[label] for label in true_columns]

threshold_vector = np.array(threshold_vector)

y_pred_micromacro = (y_proba_val >= threshold_vector).astype(int)

# Calculate the macro F1 score and micro F1 score
macro_f1 = f1_score(y_true_val, y_pred_micromacro, average='macro', zero_division=0)
micro_f1 = f1_score(y_true_val, y_pred_micromacro, average='micro', zero_division=0)

print("Micromacro macro-f1 score:", macro_f1)
print("Micromacro micro-f1 score:", micro_f1)

micromacro_report = classification_report(
    y_true_val,
    y_pred_micromacro,
    target_names= [label.replace("true_", "") for label in true_columns],
    zero_division= 0,
    output_dict= True
)

print(micromacro_report)

Micromacro macro-f1 score: 0.3313634078339961
Micromacro micro-f1 score: 0.3841059602649007
{'Meaninglessness': {'precision': 0.2549019607843137, 'recall': 0.38235294117647056, 'f1-score': 0.3058823529411765, 'support': 34.0}, 'Loneliness': {'precision': 0.42857142857142855, 'recall': 0.5217391304347826, 'f1-score': 0.47058823529411764, 'support': 23.0}, 'Death Anxiety': {'precision': 0.4418604651162791, 'recall': 0.8636363636363636, 'f1-score': 0.5846153846153846, 'support': 22.0}, 'Death Acceptance': {'precision': 0.2727272727272727, 'recall': 0.2727272727272727, 'f1-score': 0.2727272727272727, 'support': 11.0}, 'Identity Confusion': {'precision': 0.2, 'recall': 0.25, 'f1-score': 0.2222222222222222, 'support': 12.0}, 'Freedom Responsibility': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 7.0}, 'Engagement': {'precision': 0.4, 'recall': 0.2857142857142857, 'f1-score': 0.3333333333333333, 'support': 7.0}, 'Solitude': {'precision': 0.42857142857142855, 'recall': 0.5, 'f1